# Generación de predicciones de sentimiento — Gemma (vía API de Gemini)

Versión alternativa a la de Claude/Gemini-Flash, usando **`gemma-4-31b-it`** — un modelo open-weight servido a través de la misma API de Gemini, con **cuota diaria separada** de los modelos `gemini-*`. Útil para seguir generando predicciones el mismo día si agotaste la cuota de otro modelo.

Incorpora todas las correcciones aplicadas durante las pruebas anteriores:
- API key desde los **secretos de Colab** (no en texto plano).
- Detección automática de si el modelo admite `thinking_config` (Gemma probablemente no).
- Distinción entre rate-limit **por minuto** (esperar y reintentar) y cuota **diaria agotada**
  (parar todo el proceso — reintentar no sirve de nada hasta el día siguiente).
- Reintento con más tokens si la respuesta se corta por `MAX_TOKENS`.
- Checkpoint que **solo cuenta como "hechas" las filas con predicción exitosa** — las que
  fallan se reintentan automáticamente en la siguiente ejecución, en vez de quedar marcadas
  como completadas con un valor nulo para siempre.

### Documento necesario
`sentiment_ground_truth.csv`

### ⚠️ Nota metodológica si vienes de haber usado otro modelo antes
Si ya generaste una parte de las predicciones con `gemini-3.1-flash-lite` u otro modelo y ahora
completas el resto con Gemma, tendrás un archivo final con predicciones de dos modelos distintos.
No invalida la evaluación, pero **documéntalo en tu informe** (qué filas vinieron de cada modelo)
por transparencia metodológica.

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q google-genai tqdm pandas

## 2. Subir el archivo

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecciona sentiment_ground_truth.csv

In [ ]:
# @title 2b. (Solo si es una sesión nueva) Sube tu checkpoint anterior si lo tienes
from google.colab import files
import os

print("Si tienes un checkpoint descargado de una sesión anterior, súbelo ahora.")
print("Si esta sesión de Colab nunca se desconectó, puedes saltarte esta celda.")
checkpoint_upload = files.upload()
if checkpoint_upload:
    nombre_subido = list(checkpoint_upload.keys())[0]
    if nombre_subido != CHECKPOINT_PATH:
        os.rename(nombre_subido, CHECKPOINT_PATH)
    print(f"✅ Checkpoint restaurado: {CHECKPOINT_PATH}")

## 3. API key desde los secretos de Colab

Antes de ejecutar: icono de llave 🔑 en el panel izquierdo → "Añadir secreto nuevo" → nombre
`GEMINI_API_KEY` (la misma key que usaste para el modelo anterior, es la misma API) → activa el
interruptor de acceso para este notebook.

In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("✅ API key cargada desde los secretos de Colab.")
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "No encontré un secreto llamado 'GEMINI_API_KEY'. "
        "Añádelo desde el icono de llave 🔑 en el panel izquierdo de Colab."
    )
except userdata.NotebookAccessError:
    raise RuntimeError(
        "El secreto 'GEMINI_API_KEY' existe pero no le has dado acceso a este notebook. "
        "Actívalo desde el icono de llave 🔑."
    )

import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

## 4. Configuración

⚠️ Antes de lanzar el batch completo, revisa el límite real de **peticiones por minuto** para
`gemma-4-31b-it` en tu cuenta en [aistudio.google.com/rate-limit](https://aistudio.google.com/rate-limit)
y ajusta `REQUESTS_PER_MINUTE_LIMIT` acorde. El valor de abajo (10) es conservador por defecto.

In [ ]:
GROUND_TRUTH_PATH = "sentiment_ground_truth.csv"
OUTPUT_PATH = "predicciones_sentimiento_gemma.json"
CHECKPOINT_PATH = "predicciones_sentimiento_gemma_checkpoint.json"

MODEL = "gemma-4-31b-it"
MAX_OUTPUT_TOKENS = 1024  # margen amplio; se duplica automáticamente si aún así no alcanza

CHECKPOINT_EVERY = 25
MAX_RETRIES = 5
BASE_BACKOFF_SECONDS = 2

REQUESTS_PER_MINUTE_LIMIT = 25  # AJUSTA esto según lo que veas en aistudio.google.com/rate-limit
MIN_SECONDS_BETWEEN_CALLS = 60 / REQUESTS_PER_MINUTE_LIMIT * 1.1  # +10% margen de seguridad

## 5. Cliente, prompt del sistema, y detección de compatibilidad con `thinking_config`

In [ ]:
from google import genai
from google.genai import types
from google.genai import errors

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

SYSTEM_PROMPT = """Eres un analista financiero senior, experto en interpretar el impacto de noticias en los mercados.
Tu tarea es clasificar el sentimiento de mercado de una noticia para un activo/ticker concreto.

Responde ÚNICAMENTE con un objeto JSON, sin texto adicional antes ni después, con este formato exacto:
{"sentimiento": "bullish|bearish|sideways", "confianza": 0.0}

Reglas:
- "sentimiento" debe ser exactamente una de estas tres palabras: bullish, bearish, sideways.
- "confianza" es tu propia estimación (0.0 a 1.0) de cuán seguro estás de esa clasificación.
- No incluyas explicaciones, razonamiento ni texto fuera del JSON."""

BASE_CONFIG_KWARGS = dict(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.0,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    response_mime_type="application/json",
)

def build_config(with_thinking_budget: bool, max_output_tokens: int = None):
    kwargs = dict(BASE_CONFIG_KWARGS)
    if max_output_tokens is not None:
        kwargs["max_output_tokens"] = max_output_tokens
    if with_thinking_budget:
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
    return types.GenerateContentConfig(**kwargs)


print(f"Comprobando compatibilidad de '{MODEL}' con thinking_config...")
try:
    client.models.generate_content(model=MODEL, contents="Responde solo: ok", config=build_config(True))
    SUPPORTS_THINKING_CONFIG = True
    print("✅ El modelo admite thinking_config (desactivado con budget=0).")
except errors.ClientError as e:
    if "thinking" in e.message.lower():
        SUPPORTS_THINKING_CONFIG = False
        print("ℹ️  Este modelo no soporta thinking_config — se omite (normal en modelos Gemma).")
    else:
        raise

GENERATION_CONFIG = build_config(SUPPORTS_THINKING_CONFIG)

## 6. Prueba rápida — una sola llamada antes del batch completo

In [ ]:
test_prompt = """Ticker: ^GSPC
Sector/Índice: S&P 500
Fecha: 2020-03-23
Titular de la noticia: "36 Hours of Alarm and Action as Crisis Spiraled"

Clasifica el sentimiento de mercado de esta noticia para ^GSPC."""

test_response = client.models.generate_content(model=MODEL, contents=test_prompt, config=GENERATION_CONFIG)
print("Respuesta cruda:", test_response.text)

## 7. Construcción del prompt y parseo de la respuesta

In [ ]:
import re
import json

def build_user_prompt(row):
    return f"""Ticker: {row['ticker']}
Sector/Índice: {row['indice_sector']}
Fecha: {row['fecha']}
Titular de la noticia: "{row['titulo']}"

Clasifica el sentimiento de mercado de esta noticia para {row['ticker']}."""


def parse_response(raw_text):
    if raw_text is None:
        return None, None
    try:
        cleaned = raw_text.strip()
        cleaned = re.sub(r"^```(json)?|```$", "", cleaned, flags=re.MULTILINE).strip()
        data = json.loads(cleaned)
        sentimiento = str(data.get("sentimiento", "")).strip().lower()
        confianza = data.get("confianza", None)
        confianza = float(confianza) if confianza is not None else None
        return sentimiento, confianza
    except (json.JSONDecodeError, ValueError, AttributeError):
        pass

    lower = raw_text.lower()
    for kw in ["bullish", "alcista", "bearish", "bajista", "sideways", "lateral"]:
        if kw in lower:
            return kw, None
    return None, None

## 8. Llamada con throttle, backoff, y distinción cuota diaria vs. por minuto

In [ ]:
import time
import random

class DailyQuotaExhausted(Exception):
    """La cuota agotada es DIARIA, no por minuto — reintentar no sirve de nada hasta mañana."""
    pass


RETRYABLE_CODES = {429, 500, 503}
_last_call_time = [0.0]


def extract_suggested_wait(message):
    match = re.search(r"retry in (\d+(\.\d+)?)s", message, flags=re.IGNORECASE)
    return float(match.group(1)) if match else None


def is_daily_quota_error(message):
    return "PerDay" in message or "GenerateRequestsPerDay" in message


def throttle():
    elapsed = time.time() - _last_call_time[0]
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)
    _last_call_time[0] = time.time()


def call_llm_with_backoff(user_prompt):
    token_budget = MAX_OUTPUT_TOKENS
    for attempt in range(MAX_RETRIES):
        throttle()
        config = GENERATION_CONFIG.model_copy(update={"max_output_tokens": token_budget})
        try:
            response = client.models.generate_content(model=MODEL, contents=user_prompt, config=config)
            if response.text is None:
                finish_reason = response.candidates[0].finish_reason if response.candidates else None
                if finish_reason == types.FinishReason.MAX_TOKENS:
                    token_budget *= 2
                    print(f"  Sin espacio para responder (MAX_TOKENS). Reintentando con {token_budget} tokens...")
                    continue
                raise RuntimeError(f"Respuesta vacía del modelo (finish_reason={finish_reason})")
            return response.text
        except (errors.ClientError, errors.ServerError) as e:
            if e.code not in RETRYABLE_CODES:
                raise RuntimeError(f"Error no recuperable ({e.code}): {e.message}") from e

            if e.code == 429 and is_daily_quota_error(e.message):
                raise DailyQuotaExhausted(e.message)

            suggested_wait = extract_suggested_wait(e.message)
            wait = suggested_wait if suggested_wait else BASE_BACKOFF_SECONDS * (2 ** attempt)
            wait += random.uniform(0, 1)
            etiqueta = "Límite de cuota" if e.code == 429 else "Servidor saturado (temporal, no es cuota)"
            print(f"  {etiqueta} ({e.code}). Esperando {wait:.1f}s...")
            time.sleep(wait)
    raise RuntimeError(f"Fallaron los {MAX_RETRIES} reintentos para esta fila.")

## 9. Checkpointing

**Corrección importante frente a la versión anterior**: el checkpoint ahora solo guarda las filas
con predicción **exitosa**. Las que fallan NO se marcan como procesadas, así que la próxima vez
que ejecutes el bucle se reintentan automáticamente — antes quedaban con `sentimiento_predicho: null`
guardado para siempre, y el script las daba por "hechas" sin haber conseguido nunca una respuesta.

In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            data = json.load(f)
        # Solo cuentan como "procesadas" las filas con predicción exitosa (no None)
        exitosas = [r for r in data if r["sentimiento_predicho"] is not None]
        ids_procesados = {r["id"] for r in exitosas}
        print(f"Checkpoint encontrado: {len(exitosas)} filas exitosas ya procesadas, se reanuda desde ahí.")
        return exitosas, ids_procesados
    return [], set()


def save_checkpoint(resultados):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

## 10. Bucle principal

Si la cuota diaria se agota, el bucle **se detiene por completo** (no sigue fallando fila a fila)
y guarda el progreso — vuelve a ejecutar esta misma celda mañana, o cambia `MODEL` a otro con
cuota separada para seguir hoy.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

df = pd.read_csv(GROUND_TRUTH_PATH)
resultados, ids_procesados = load_checkpoint()
pendientes = df[~df["id"].isin(ids_procesados)]
print(f"Total filas: {len(df)} | Ya procesadas con éxito: {len(ids_procesados)} | Pendientes: {len(pendientes)}")

detenido_por_cuota_diaria = False

for i, (_, row) in enumerate(tqdm(pendientes.iterrows(), total=len(pendientes), desc="Clasificando sentimiento (Gemma)")):
    user_prompt = build_user_prompt(row)
    try:
        raw_text = call_llm_with_backoff(user_prompt)
        sentimiento, confianza = parse_response(raw_text)
        resultados.append({
            "id": int(row["id"]), "sentimiento_predicho": sentimiento,
            "confianza": confianza, "respuesta_cruda": raw_text,
        })
    except DailyQuotaExhausted:
        save_checkpoint(resultados)
        print(f"\n🛑 Cuota DIARIA agotada en la fila id={row['id']}.")
        print(f"   Progreso guardado: {len(resultados)} filas exitosas.")
        print("   Vuelve a ejecutar esta celda mañana (retomará solo lo pendiente), o cambia MODEL.")
        detenido_por_cuota_diaria = True
        break
    except RuntimeError as e:
        print(f"  ⚠️ Fila id={row['id']} falló tras reintentos: {e}")
        # No se añade a 'resultados' -> no cuenta como procesada -> se reintentará la próxima vez

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(resultados)
        if (i + 1) % (CHECKPOINT_EVERY * 4) == 0:  # cada ~100 filas, descarga automática
            from google.colab import files as colab_files
            colab_files.download(CHECKPOINT_PATH)

save_checkpoint(resultados)
print(f"\nProgreso actual: {len(resultados)} de {len(df)} filas procesadas con éxito.")
if detenido_por_cuota_diaria:
    print("(Proceso detenido por cuota diaria agotada, no completado.)")

## 11. Guardar y descargar el archivo final

El archivo final incluye explícitamente, con valor `null`, cualquier fila que siga sin predicción
exitosa tras esta ejecución — para que `evaluacion_sentimiento_RAG.ipynb` sepa que existen (aunque
las descarta al calcular métricas) y para que puedas ver de un vistazo cuántas faltan.

In [ ]:
ids_con_prediccion = {r["id"] for r in resultados}
resultados_completos = list(resultados)
for id_faltante in df.loc[~df["id"].isin(ids_con_prediccion), "id"]:
    resultados_completos.append({
        "id": int(id_faltante), "sentimiento_predicho": None,
        "confianza": None, "respuesta_cruda": None,
    })
resultados_completos.sort(key=lambda r: r["id"])

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados_completos, f, ensure_ascii=False, indent=2)

n_exitosas = sum(1 for r in resultados_completos if r["sentimiento_predicho"] is not None)
print(f"Guardado {OUTPUT_PATH}: {n_exitosas} de {len(resultados_completos)} filas con predicción.")

from google.colab import files as colab_files
colab_files.download(OUTPUT_PATH)